# Fused Pair-Feature Bias in Flash Attention

March 2026

## The Problem: Pair-Wise Bias in Attention

Models occasionally need a **pair-wise bias** that encodes e.g. geometric relationships between tokens (residues, atoms, etc.):

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d}} + \underbrace{B}_{\text{pair bias}}\right) V$$

where $B_{j,i,h}$ depends on features of tokens $i$ and $j$ — e.g. distances, angles, or other spatial relationships.

**Examples:**
- **AlphaFold-style**: pair representations projected to per-head biases
- **Distance-based**: $\phi(x_i, x_j)$ = directional RBFs, euclidean distance, inverse distance
- **Feature interactions**: elementwise products, differences, concatenation → linear projection

The bias is a function of **pair features** projected to heads:

$$B_{j,i,h,b} = \sum_{p=1}^{d_{pf}} W_{h,p} \cdot \phi_p\!\left(\mathbf{f}_i^{(b)},\, \mathbf{f}_j^{(b)}\right)$$

where $\phi: \mathbb{R}^F \times \mathbb{R}^F \to \mathbb{R}^{d_{pf}}$ is a pair feature function, and $W \in \mathbb{R}^{H \times d_{pf}}$ projects to heads.

## The Memory Wall

The pair bias tensor $B$ has shape $(H, L, L, B)$ — it's **quadratic in sequence length**.

| $L$ | $H$ | $B$ | `sizeof(B)` Float32 | + backward (`dB`) |
|-----|-----|-----|---------------------|-------------------|
| 256 | 8 | 4 | 8 MiB | 16 MiB |
| 1024 | 8 | 4 | 128 MiB | 256 MiB |
| 4096 | 4 | 4 | **1 GiB** | **2 GiB** |
| 4096 | 8 | 4 | **2 GiB** | **4 GiB** |

This dominates memory for any model that uses pair-wise attention at moderate sequence lengths.

## Flash Attention Recap

[Dao et al., 2022](https://arxiv.org/abs/2205.14135) — the key insight is **tiling** the attention computation:

```py
for each Q-tile (block of query rows):
    for each K-tile (block of key rows):
        load Q_tile, K_tile, V_tile into shared memory
        S_tile = Q_tile @ K_tile^T / sqrt(d)     ← tile_size × tile_size
        update running softmax + output accumulator
```

The attention matrix is never fully materialized — only one **tile** (e.g. $32 \times 32$) exists at a time in shared memory/registers.

**Result:** $O(1)$ extra memory instead of $O(L^2)$ for the attention matrix.

Flash attention can trivially support **additive bias** — if you pass a precomputed $B$, the kernel loads the corresponding tile from HBM:

```py
S_tile += B[h, q_start:q_end, k_start:k_end, b]    ← one HBM load per tile
```

But this still requires materializing and storing the full $(H, L, L, B)$ bias tensor.

## The Key Insight: Fuse Pair Features Into the Tile Loop

Instead of precomputing $B$ and reading it from HBM, **compute each tile of $B$ on the fly** from per-token features:

```py
for each Q-tile:
    load q_features[tile_rows] into shared memory       ← (F, tile_size), tiny
    for each K-tile:
        load Q_tile, K_tile, V_tile into shared memory
        load k_features[tile_cols] into shared memory    ← (F, tile_size), tiny
        S_tile = Q_tile @ K_tile^T / sqrt(d)
        for each (i, j) in tile:
            phi = pair_feature_tuple(op, q_feat[i], k_feat[j])    ← in registers
            S_tile[i,j] += dot(pair_proj[h, :], phi)              ← d_pf MADs
        update running softmax + output accumulator
```

**What changed:**
- Features are $(F, L, B)$ — **linear** in $L$, loaded once per tile
- The pair feature function $\phi$ runs **in registers** per $(i, j)$ — no allocation
- `pair_proj` is $(H, d_{pf})$ — tiny, lives in registers/L1
- The $L \times L$ bias tensor **never exists in memory**

**Memory:** $O(F \cdot L \cdot B)$ instead of $O(H \cdot L^2 \cdot B)$ — a reduction by factor $\frac{H \cdot L}{F}$

At $L{=}4096, H{=}4, F{=}4$: factor of **4096×** less memory for the bias.

## Why Not Just Fuse the Materialization?

A natural question: couldn't you write a single CUDA kernel that computes $B$ from features, avoiding the broadcast intermediates, then pass it to flash attention?

Yes, but you still need to **store** $B$ in HBM:
- Forward: write $(H, L, L, B)$ to HBM, flash attention reads it back → **2× memory bandwidth** for the bias
- Backward: store $\frac{\partial \mathcal{L}}{\partial B}$ of the same size → another $(H, L, L, B)$ allocation

The fused approach avoids both: the bias exists only in shared memory / registers, for one tile at a time.

| Approach | Forward mem | Backward mem | Bias bandwidth |
|----------|------------|-------------|----------------|
| Naive broadcast | $O(k \cdot H L^2 B)$ | $O(k \cdot H L^2 B)$ | N/A (many passes) |
| Fused materialize + flash attn | $O(H L^2 B)$ | $O(2 \cdot H L^2 B)$ | 2× (write + read) |
| **Fused pair features (ours)** | $O(F L B)$ | $O(F L B)$ | **0** (never in HBM) |

## The User-Facing API

Users define a pair feature op as an `isbits` struct with a few methods:

```julia
struct MyPairOp{T}
    gamma::T
end

# How many pair features does phi produce?
NNop.pair_feature_dim(::MyPairOp) = 3

# The pair feature function — runs on GPU, per (i, j) pair
# qvals/kvals are NTuples of length F = size(q_features, 1)
@inline function NNop.pair_feature_tuple(op::MyPairOp{T}, q::NTuple{2,T}, k::NTuple{2,T}) where T
    dx = q[1] - k[1]
    dy = q[2] - k[2]
    return (dx, dy, sqrt(dx^2 + dy^2 + 1f-4))
end

# Optional: VJP for gradients w.r.t. features (only needed if feature_grads=true)
@inline function NNop.pair_feature_tuple_pullback(op::MyPairOp{T}, q, k, dphi)
    # ... return (dq, dk)
end
```

Then call flash attention with the pair op:

```julia
o = NNop.flash_attention(q, k, v, q_features, k_features, pair_proj, pair_op; causal=false)
```

Gradients through `q`, `k`, `v`, and `pair_proj` are always computed. Gradients through `q_features`/`k_features` are opt-in via `feature_grads=true`.

## Design Decisions

**Why NTuples instead of arrays?**
- Tuples live in **registers**, not memory. No shared memory bank conflicts, no sync barriers.
- The pair feature dimension $d_{pf}$ is a compile-time constant → the compiler fully unrolls everything.

**Why are feature gradients opt-in?**
- Computing $\frac{\partial \mathcal{L}}{\partial \mathbf{f}_i}$ requires the VJP of $\phi$ and extra register pressure in the backward kernel.
- Many use cases have features that are **input data** (e.g. atom coordinates from a PDB) — no gradient needed.
- When features come from an earlier network layer, turn on `feature_grads=true`.

**Why separate `pair_proj` from the pair op?**
- The pair features $\phi(\mathbf{f}_i, \mathbf{f}_j)$ are **head-independent** — computed once.
- The projection $W \in \mathbb{R}^{H \times d_{pf}}$ adds the head dimension cheaply.
- Gradients for $W$ come essentially for free (accumulated in the backward kernel).

**Dense feature loading:**
- All `F = size(q_features, 1)` features are loaded into shared memory per tile.
- If you only need a subset (e.g. spatial coords from a larger feature vector), just pass the slice you want.

## Demo: Directional LeakyReLU + Euclidean Distance

A concrete pair op for molecular/structural modeling with 4 input features → 9 pair features:

For features $\mathbf{f} = (x, y, z, \text{idx})$:

$$\phi(\mathbf{f}_i, \mathbf{f}_j) = \Big(\text{lrelu}(d_1),\; \text{lrelu}(-d_1),\; \text{lrelu}(d_2),\; \text{lrelu}(-d_2),\; \text{lrelu}(d_3),\; \text{lrelu}(-d_3),\; \text{lrelu}(d_4),\; \text{lrelu}(-d_4),\; r\Big)$$

where $d_f = f_i^{(f)} - f_j^{(f)}$ and $r = \sqrt{d_1^2 + d_2^2 + d_3^2 + \epsilon}$ (euclidean distance on spatial coords only).

In [4]:
using NNop, CUDA, Zygote, Random

In [7]:
struct LeakyReLUDist{T}
    slope::T
    eps::T
end

NNop.pair_feature_dim(::LeakyReLUDist) = 9

@inline lrelu(s::T, x::T) where T = ifelse(x > 0, x, s * x)
@inline dlrelu(s::T, x::T) where T = ifelse(x > 0, one(T), s)

@inline function NNop.pair_feature_tuple(op::LeakyReLUDist, q::NTuple{4,T}, k::NTuple{4,T}) where T
    d = d1, d2, d3, _ = q .- k
    s = op.slope
    r = √(d1*d1 + d2*d2 + d3*d3 + op.eps)
    return (lrelu.(s, d)..., lrelu.(s, .-d)..., r)
end

@inline function NNop.pair_feature_tuple_pullback(op::LeakyReLUDist, q::NTuple{4,T}, k::NTuple{4,T}, dp::NTuple{9,T}) where T
    d1, d2, d3, d4 = q .- k
    s = op.slope
    inv_r = inv(sqrt(d1*d1 + d2*d2 + d3*d3 + op.eps))
    dq1 = dp[1]*dlrelu(s,d1) - dp[5]*dlrelu(s,-d1) + dp[9]*d1*inv_r
    dq2 = dp[2]*dlrelu(s,d2) - dp[6]*dlrelu(s,-d2) + dp[9]*d2*inv_r
    dq3 = dp[3]*dlrelu(s,d3) - dp[7]*dlrelu(s,-d3) + dp[9]*d3*inv_r
    dq4 = dp[4]*dlrelu(s,d4) - dp[8]*dlrelu(s,-d4)
    return (dq1,dq2,dq3,dq4), (-dq1,-dq2,-dq3,-dq4)
end

pair_op = LeakyReLUDist(0.1f0, 1f-4)
println("Pair op: 4 input features → $(NNop.pair_feature_dim(pair_op)) pair features")

Pair op: 4 input features → 9 pair features


In [8]:
# Naive materialization for reference
function materialize_pair_bias(q_features, k_features, pair_proj, pair_op)
    F, L, B = size(q_features)
    QH = size(pair_proj, 1)
    T = eltype(q_features)
    bias = CUDA.zeros(T, QH, L, L, B)
    qf = Array(q_features)
    kf = Array(k_features)
    pp = Array(pair_proj)
    bias_cpu = zeros(T, QH, L, L, B)
    for b in 1:B, j in 1:L, i in 1:L
        qvals = ntuple(f -> qf[f, i, b], Val(F))
        kvals = ntuple(f -> kf[f, j, b], Val(F))
        phi = NNop.pair_feature_tuple(pair_op, qvals, kvals)
        for h in 1:QH
            bias_cpu[h, i, j, b] = sum(pp[h, p] * phi[p] for p in 1:length(phi))
        end
    end
    copyto!(bias, bias_cpu)
    return bias
end

materialize_pair_bias (generic function with 1 method)

In [9]:
# Small demo — correctness check
Random.seed!(42)
E, L, QH, KVH, B, F = 64, 128, 4, 4, 2, 4

q = CUDA.randn(Float32, E, L, QH, B)
k = CUDA.randn(Float32, E, L, KVH, B)
v = CUDA.randn(Float32, E, L, KVH, B)
q_features = CUDA.randn(Float32, F, L, B)
k_features = CUDA.randn(Float32, F, L, B)
pair_proj = CUDA.randn(Float32, QH, 9)

# Materialized path: compute bias explicitly, then standard flash attention
pair_bias = materialize_pair_bias(q_features, k_features, pair_proj, pair_op)
o_materialized = NNop.flash_attention(q, k, v, pair_bias; causal=false)

# Fused path: pair features computed on-the-fly inside the kernel
o_fused = NNop.flash_attention(q, k, v, q_features, k_features, pair_proj, pair_op; causal=false)

max_err = maximum(abs.(Array(o_materialized) .- Array(o_fused)))
println("Max absolute error: $(max_err)")
println("Match: $(max_err < 1e-4 ? "✓" : "✗")")

println("\nMemory comparison:")
println("  Materialized bias tensor: $(Base.format_bytes(sizeof(pair_bias)))")
println("  Features (q+k):           $(Base.format_bytes(sizeof(q_features) + sizeof(k_features)))")
println("  Pair projection:          $(Base.format_bytes(sizeof(pair_proj)))")
println("  Ratio: $(round(sizeof(pair_bias) / (sizeof(q_features) + sizeof(k_features) + sizeof(pair_proj)); digits=1))×")

Max absolute error: 1.4305115e-6
Match: ✓

Memory comparison:
  Materialized bias tensor: 512.000 KiB
  Features (q+k):           8.000 KiB
  Pair projection:          144 bytes
  Ratio: 62.9×


In [ ]:
# Gradient correctness — fused backward matches materialized
grad_mat = Zygote.gradient(q, k, v) do q, k, v
    sum(NNop.flash_attention(q, k, v, pair_bias; causal=false))
end
grad_fused = Zygote.gradient(q, k, v, pair_proj) do q, k, v, pp
    sum(NNop.flash_attention(q, k, v, q_features, k_features, pp, pair_op; causal=false))
end

println("Gradient errors (fused vs materialized):")
for (name, i) in [("dq", 1), ("dk", 2), ("dv", 3)]
    err = maximum(abs.(Array(grad_mat[i]) .- Array(grad_fused[i])))
    println("  $name max error: $(round(err; sigdigits=3))")
end
println("  dproj norm: $(round(sqrt(sum(Array(grad_fused[4]).^2)); sigdigits=4))")

## Benchmark Results

**Setup:** D=64, L=4096, H=4, B=4, Float32, DirLeakyReLU+Distance (9 pair features), RTX PRO 6000 (Blackwell)

Pair bias tensor at this size: **1 GiB**

### Precomputed bias vs Fused (fair comparison)

The pair bias is precomputed and passed directly to flash attention — no broadcast intermediates.

| | Time | GPU Memory Allocated |
|---|---:|---:|
| **Precomputed bias → FWD** | 9.1 ms | 16.5 MiB |
| **Fused features → FWD** | 10.0 ms | 16.5 MiB |
| **Precomputed bias → FWD+BWD** | 479 ms | **1.09 GiB** |
| **Fused → FWD+BWD** (proj grads only) | 542 ms | **97 MiB** |
| **Fused → FWD+BWD** (all grads) | 702 ms | **99 MiB** |

### Key observations

- **Forward:** Nearly identical speed. The fused kernel does extra compute (pair features per tile) but avoids an HBM read for the 1 GiB bias tensor — these roughly cancel.

- **Backward:** The precomputed path must allocate $\frac{\partial \mathcal{L}}{\partial B}$ — another 1 GiB tensor. The fused path doesn't: **11× less memory**.

- **Speed tradeoff:** Fused backward is ~13% slower (542 vs 479 ms) due to recomputing pair features. With `feature_grads=true` it's ~47% slower due to the VJP computation.

- **The real win is enabling larger $L$:** At $L{=}4096$ you save 1 GiB. At $L{=}8192$ you'd save 4 GiB (pair bias = $4 \times 8192^2 \times 4 \times 4$ bytes = 4 GiB). The fused path stays at $O(L)$.

## Summary

**What we built:** A generic extension point in NNop.jl's flash attention kernel that fuses arbitrary user-defined pair feature functions into the tiled attention loop.

**How it works:**
1. Per-token features $(F, L, B)$ are loaded into shared memory per tile — $O(L)$ memory
2. Pair features $\phi(f_i, f_j)$ are computed in registers per $(i, j)$ — no allocation
3. A learned projection $(H, d_{pf})$ maps pair features to per-head bias — tiny
4. The $(H, L, L, B)$ bias tensor **never exists in memory**

**What you get:**
- **11× less GPU memory** for backward pass vs precomputed bias at $L{=}4096$
- **Scales to much larger $L$** without OOM (linear vs quadratic memory)
- ~13% slower backward (compute vs memory tradeoff)
- Full autodiff support: gradients for $Q, K, V$, projection $W$, and optionally input features
- Any `isbits` pair function — users just implement a struct + 3-4 methods

**Implementation:** ~820 lines of Julia using KernelAbstractions.jl. Works on CUDA and AMDGPU. Supports GQA, causal masking, variable-length sequences, FP16/BF16/FP32.